In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np

def smooth_data(scalars, factor=0.9):
    """
    Smoothing with Bias Correction.
    - factor: 0.0 to 1.0 (Higher = Smoother, 0.9 is standard for GANs)
    """
    if not scalars:
        return []

    def pass_filter(data):
        last = 0
        res = []
        for i, point in enumerate(data):
            # 1. Standard Exponential Moving Average
            last = last * factor + (1 - factor) * point
            
            # 2. Bias Correction 
            # This "boosts" the early values so they aren't stuck at 0 
            # or anchored to the first outlier.
            correction = 1 - (factor ** (i + 1))
            res.append(last / correction)
        return res

    # Pass 1: Forward (Removes jitter)
    forward = pass_filter(scalars)
    
    # Pass 2: Backward (Removes the 'time lag' so peaks align)
    backward = pass_filter(forward[::-1])[::-1]

    return backward

def parse_pix2pix_log(file_path):
    epochs, g_gan, g_l1, d_total = [], [], [], []
    pattern = r"epoch: (\d+).*?G_GAN: ([\d\.]+).*?G_L1: ([\d\.]+).*?D_real: ([\d\.]+).*?D_fake: ([\d\.]+)"
    
    with open(file_path, 'r') as f:
        for line in f:
            match = re.search(pattern, line)
            if match:
                epochs.append(int(match.group(1)))
                g_gan.append(float(match.group(2)))
                g_l1.append(float(match.group(3)))
                d_total.append(float(match.group(4)) + float(match.group(5)))
    return epochs, g_gan, g_l1, d_total

def plot_model(log, name="Model", smooth_weight=0.95):
    data = parse_pix2pix_log(log)
    epochs = data[0]

    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    metrics = [("G_GAN Loss", 1), ("G_L1 Loss", 2), ("Total D Loss", 3)]

    for i, (label, idx) in enumerate(metrics):
        raw_y = data[idx]
        smoothed_y = smooth_data(raw_y, factor=smooth_weight)
        
        # Background raw data
        #ax[i].plot(epochs, raw_y, color='gray', alpha=0.2)
        # Foreground smoothed data
        ax[i].plot(epochs, smoothed_y, color='tab:blue', linewidth=2)
        
        ax[i].set_title(label, fontsize=14)
        ax[i].set_xlabel("Epoch", fontsize=12)
        ax[i].legend()

    plt.tight_layout()
    plt.savefig(f"loss_curves_{name}.svg", format="svg", bbox_inches="tight")
    plt.show()

def compare_models(log1, log2, name1="Model A", name2="Model B", smooth_weight=0.99):
    datasets = [
        (parse_pix2pix_log(log1), name1, 'tab:blue'),
        (parse_pix2pix_log(log2), name2, 'tab:orange')
    ]

    fig, ax = plt.subplots(1, 3, figsize=(15, 3))
    metrics = [("G_GAN Loss", 1), ("G_L1 Loss", 2), ("Total D Loss", 3)]

    for i, (label, idx) in enumerate(metrics):
        for data, model_name, color in datasets:
            raw_y = data[idx]
            smoothed_y = smooth_data(raw_y, factor=smooth_weight)
            
            # Plot raw data with high transparency
            #ax[i].plot(data[0], raw_y, color=color, alpha=0.2)
            # Plot smoothed data bold
            ax[i].plot(data[0], smoothed_y, color=color, label=model_name, linewidth=2)
            
        ax[i].set_title(label)
        ax[i].set_xlabel("Epoch")
        ax[i].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_model("")